# 07 Evaluate WLASL2000 Selected Models

## What this notebook does
This notebook compares the trained WLASL2000 models:

- Light V2 from scratch
- Light V2 fine-tuned from WLASL1000
- optional small Transformer

## Why this matters
The final app should use the model that is most reliable for deployment, not only the model with the highest raw score.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Set paths

In [ ]:
PROJECT_ROOT = Path("E:/Be_My_Ear")

DATASET_NAME = "WLASL2000"
PREFIX = "wlasl2000"

MODEL_DIR = PROJECT_ROOT / "models" / "ASL" / DATASET_NAME
REPORT_DIR = PROJECT_ROOT / "reports" / f"phase1_{PREFIX}"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Model folder:", MODEL_DIR)
print("Report folder:", REPORT_DIR)

## 2. Load model result summaries

In [ ]:
result_files = [
    MODEL_DIR / f"light_v2_scratch_{PREFIX}_result_summary.csv",
    MODEL_DIR / f"light_v2_finetuned_from_wlasl1000_{PREFIX}_result_summary.csv",
    MODEL_DIR / f"small_transformer_{PREFIX}_result_summary.csv",
]

rows = []

for file in result_files:
    if file.exists():
        temp = pd.read_csv(file)
        if len(temp) > 0:
            row = temp.iloc[0].to_dict()
            row["source_file"] = str(file)
            rows.append(row)
    else:
        print("Missing:", file)

if not rows:
    raise FileNotFoundError("No WLASL2000 result summaries found yet.")

comparison_df = pd.DataFrame(rows)

comparison_df

## 3. Clean and save comparison table

In [ ]:
columns = [
    "dataset",
    "model",
    "clean_samples",
    "classes",
    "test_samples",
    "input_shape",
    "test_top1_accuracy",
    "test_top3_accuracy",
    "test_top5_accuracy",
    "test_macro_f1",
    "best_val_f1",
    "best_val_top5",
    "checkpoint_epoch",
    "model_path",
    "source_file"
]

available_columns = [col for col in columns if col in comparison_df.columns]
clean_df = comparison_df[available_columns].copy()

metric_cols = [
    "test_top1_accuracy",
    "test_top3_accuracy",
    "test_top5_accuracy",
    "test_macro_f1",
    "best_val_f1",
    "best_val_top5"
]

for col in metric_cols:
    if col in clean_df.columns:
        clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

clean_df = clean_df.sort_values("test_top1_accuracy", ascending=False)

comparison_file = REPORT_DIR / f"{PREFIX}_selected_model_comparison.csv"
clean_df.to_csv(comparison_file, index=False)

print("Saved comparison:", comparison_file)

clean_df

## 4. Plot Top-1, Top-3, and Top-5 accuracy

In [ ]:
plot_df = clean_df.set_index("model")

plt.figure(figsize=(10, 5))
plot_df[["test_top1_accuracy", "test_top3_accuracy", "test_top5_accuracy"]].plot(kind="bar")
plt.title("WLASL2000 Accuracy Comparison")
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 5. Plot Macro F1

In [ ]:
plt.figure(figsize=(8, 5))
plot_df["test_macro_f1"].plot(kind="bar")
plt.title("WLASL2000 Macro F1 Comparison")
plt.xlabel("Model")
plt.ylabel("Macro F1")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 6. Load confidence threshold reports

In [ ]:
threshold_files = {
    "Light V2 Scratch": REPORT_DIR / f"light_v2_scratch_{PREFIX}_confidence_threshold_analysis.csv",
    "Light V2 Fine-tuned": REPORT_DIR / f"light_v2_finetuned_from_wlasl1000_{PREFIX}_confidence_threshold_analysis.csv",
    "Transformer": REPORT_DIR / f"small_transformer_{PREFIX}_confidence_threshold_analysis.csv",
}

threshold_rows = []

for model_name, file in threshold_files.items():
    if file.exists():
        temp = pd.read_csv(file)
        temp["model"] = model_name
        threshold_rows.append(temp)
    else:
        print("Missing threshold file:", file)

if threshold_rows:
    threshold_df = pd.concat(threshold_rows, ignore_index=True)
    threshold_comparison_file = REPORT_DIR / f"{PREFIX}_confidence_threshold_model_comparison.csv"
    threshold_df.to_csv(threshold_comparison_file, index=False)

    print("Saved confidence threshold comparison:", threshold_comparison_file)
    display(threshold_df.head())
else:
    threshold_df = pd.DataFrame()
    print("No threshold reports found.")

## 7. Compare deployment thresholds

In [ ]:
if len(threshold_df) > 0:
    deployment_thresholds = threshold_df[
        threshold_df["confidence_threshold"].isin([0.5, 0.7])
    ].copy()

    deployment_thresholds = deployment_thresholds[
        [
            "model",
            "confidence_threshold",
            "coverage",
            "top1_accuracy_on_confident_samples",
            "top5_accuracy_on_confident_samples",
            "num_confident_samples"
        ]
    ]

    deployment_thresholds = deployment_thresholds.sort_values(
        ["confidence_threshold", "top1_accuracy_on_confident_samples"],
        ascending=[True, False]
    )

    deployment_threshold_file = REPORT_DIR / f"{PREFIX}_deployment_threshold_comparison.csv"
    deployment_thresholds.to_csv(deployment_threshold_file, index=False)

    print("Saved deployment threshold comparison:", deployment_threshold_file)
    display(deployment_thresholds)
else:
    print("No threshold comparison available.")

## 8. Select recommended deployment model

In [ ]:
best_top1 = clean_df.sort_values("test_top1_accuracy", ascending=False).iloc[0]
best_top5 = clean_df.sort_values("test_top5_accuracy", ascending=False).iloc[0]
best_f1 = clean_df.sort_values("test_macro_f1", ascending=False).iloc[0]

print("WLASL2000 model selection summary")
print("---------------------------------")
print("Best Top-1:", best_top1["model"], "|", round(best_top1["test_top1_accuracy"], 4))
print("Best Top-5:", best_top5["model"], "|", round(best_top5["test_top5_accuracy"], 4))
print("Best Macro F1:", best_f1["model"], "|", round(best_f1["test_macro_f1"], 4))

print("\nRecommended app model based on Top-1:")
print(best_top1["model"])

print("\nDeployment logic:")
print("confidence >= 0.70 → speak/show Top-1")
print("0.40 <= confidence < 0.70 → show Top-5 suggestions")
print("confidence < 0.40 → ask user to sign again")